# Widdershins

A tool for assigning people to group activities over a number of sessions, where groups are periodically rotated.

An optional objective is maximising the total value of pairings of people, based on per-pair values.

For now, if you want one or more people to anchor a group, just leave them out of the list.

Named by a Scot in the Southern Hemisphere (where sunwise is widdershins), for rotating against the general direction (with new value from pairings).

In [1]:
import itertools
import numpy as np
from ortools.sat.python import cp_model

## Input data

Lists of people, groups, sessions and rotations.

Optional pairing value matrix. Specifying this for larger groups can lead to long solution times.

In [2]:
# the number of People, Groups and Sessions
# rotation at session indices
P = 8
G = 3
S = 10
rotation_at = [3, 6]

if len(rotation_at) + 1 > G:
    raise ValueError("Must have more groups than rotations")
    
if min(rotation_at) < 1 or max(rotation_at) > S - 1:
    raise ValueError("Invalid rotation index")

In [3]:
# optional pairwise value data
# this example is distance between people in a fictional org chart
# or use None to solve without optimisation
pair_values = np.array([[0, 1, 1, 2, 2, 2, 2, 2],
                        [1, 0, 1, 1, 1, 2, 2, 2],
                        [1, 1, 0, 2, 2, 1, 1, 1],
                        [2, 1, 2, 0, 1, 3, 3, 3],
                        [2, 1, 2, 1, 0, 3, 3, 3],
                        [2, 2, 1, 3, 3, 0, 1, 1],
                        [2, 2, 1, 3, 3, 1, 0, 1],
                        [2, 2, 1, 3, 3, 1, 1, 0]])

## Define constraint programming model

In [4]:
model = cp_model.CpModel()

### Define core variables

Core variables:
* assignments (the plan) - of people to groups in individual sessions
* enrolments - peeople in a given group in any session
* rotations - being in which sessions groups are rotated

In [5]:
assignments = [[[model.NewBoolVar(f"a_{p}_{s}_{g}") for g in range(G)] for s in range(S)] for p in range(P)]
enrolments = [[model.NewBoolVar(f"e_{p}_{g}") for g in range(G)] for p in range(P)]
rotations = [model.NewBoolVar(f"r_{s}") for s in range(S)]

### Define core constraints

We define the following core constraints:
* each person is assigned to exactly one group per session
* people are assigned evenly to groups each session
* people must participate in as many groups as there are rotations
* assignments cannot change between sessions if not at defined rotations

In [6]:
# each person is assigned to exactly one group per session
for s in range(S):
    for p in range(P):
        model.Add(sum(assignments[p][s]) == 1)

In [7]:
# people are assigned evenly to groups each session
min_group_size = P // G
for s in range(S):
    for g in range(G):
        model.Add(sum([assignments[p][s][g] for p in range(P)]) >= min_group_size)
        model.Add(sum([assignments[p][s][g] for p in range(P)]) <= min_group_size + 1)

In [8]:
# people must participate in as many groups as there are rotations
for p in range(P):
    for g in range(G):
        model.Add(sum([assignments[p][s][g] for s in range(S)]) >= 1).OnlyEnforceIf(enrolments[p][g])
        model.Add(sum([assignments[p][s][g] for s in range(S)]) == 0).OnlyEnforceIf(enrolments[p][g].Not())
        
for p in range(P):
    model.Add(sum([enrolments[p][g] for g in range(G)]) == len(rotation_at) + 1)

In [9]:
# assignments cannot change between sessions if not at defined rotations
for s in range(S):
    model.Add(rotations[s] == (s in rotation_at))

for s in range(1, S):
    for p in range(P):
        for g in range(G):
            model.Add(assignments[p][s][g] == assignments[p][s - 1][g]).OnlyEnforceIf(rotations[s].Not())

### Optionally define maximisation objective

The objective is to maximise the total value of all pairings over all sessions.

Note that if no objective is defined the solver will still return a solution if constraints satisfiable.

Additional pairing variables are defined to support the objective.

Additional constraints cause the pairing variables to be set consistently with the assignments.

In [10]:
if pair_values is not None:
    # all pairs of people in individual sessions, per group
    paired_assignments = [[[[model.NewBoolVar(f"pa_{p}_{q}_{s}_{g}") for g in range(G)] for s in range(S)] for q in range(P)] for p in range(P)]
    # all pairs for all sessions
    paired_sessions = [[[model.NewBoolVar(f"ps_{p}_{q}_{s}") for s in range(S)] for q in range(P)] for p in range(P)]
    # all pairs in any session
    pairings = [[model.NewBoolVar(f"p_{p}_{q}") for q in range(P)] for p in range(P)]

In [11]:
if pair_values is not None:
    model.Maximize(sum([pair_values[p, q] * pairings[p][q] for p, q in itertools.product(range(P), repeat=2)]))

In [12]:
if pair_values is not None:
    # constrain paired assigments to occur when people are in the same group in the same session
    for g, s in itertools.product(range(G), range(S)):
        for p, q in itertools.product(range(P), repeat=2):
            model.AddMinEquality(paired_assignments[p][q][s][g], [assignments[p][s][g], assignments[q][s][g]])

    # constrain paired sessions to occur if any paired assignment per session across groups
    for s in range(S):       
        for p, q in itertools.product(range(P), repeat=2):
            model.Add(sum([paired_assignments[p][q][s][g] for g in range(G)]) >= 1).OnlyEnforceIf(paired_sessions[p][q][s])
            model.Add(sum([paired_assignments[p][q][s][g] for g in range(G)]) == 0).OnlyEnforceIf(paired_sessions[p][q][s].Not())

    # constrain pairings to occur if any paired session across sessions
    for p, q in itertools.product(range(P), repeat=2):
        model.Add(sum([paired_sessions[p][q][s] for s in range(S)]) >= 1).OnlyEnforceIf(pairings[p][q])
        model.Add(sum([paired_sessions[p][q][s] for s in range(S)]) == 0).OnlyEnforceIf(pairings[p][q].Not())

## Solve

Solve and display solution. Interrupt a long-running solution to see best feasible result instead of optimal.

In [13]:
def print_assignment_solution(assg_sol):
    def group_index(group_bool_array):
        return group_bool_array.index(True)
    
    for p in range(P):
        print(f"person {p}: " + ", ".join([str(group_index(a_s)) for a_s in assg_sol[p]]))
        
def print_pairing_solution(pair_sol):
    for p in range(P):
        print(f"person {p}: " + ", ".join([str(i) for i in np.nonzero(pair_sol[p])]))        

In [14]:
solver = cp_model.CpSolver()
status = solver.solve(model)
print("solution", status)
if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print("=== Assignments ===")
    assignment_solution = [[[solver.value(assignments[p][s][g]) \
                             for g in range(G)] \
                             for s in range(S)] \
                             for p in range(P)]
    print_assignment_solution(assignment_solution)
    
    print("=== Pairings ===")
    if pair_values is not None:
        pairing_solution = [[solver.value(pairings[p][q]) \
                                 for q in range(P)] \
                                 for p in range(P)]
        print_pairing_solution(pairing_solution)
    else:
        print("enable pairing objective for pairing solution")

solution CpSolverStatus.OPTIMAL
=== Assignments ===
person 0: 2, 2, 2, 0, 0, 0, 1, 1, 1, 1
person 1: 2, 2, 2, 1, 1, 1, 0, 0, 0, 0
person 2: 1, 1, 1, 2, 2, 2, 0, 0, 0, 0
person 3: 1, 1, 1, 0, 0, 0, 2, 2, 2, 2
person 4: 0, 0, 0, 2, 2, 2, 1, 1, 1, 1
person 5: 2, 2, 2, 0, 0, 0, 1, 1, 1, 1
person 6: 0, 0, 0, 1, 1, 1, 2, 2, 2, 2
person 7: 1, 1, 1, 2, 2, 2, 0, 0, 0, 0
=== Pairings ===
person 0: [0 1 3 4 5]
person 1: [0 1 2 5 6 7]
person 2: [1 2 3 4 7]
person 3: [0 2 3 5 6 7]
person 4: [0 2 4 5 6 7]
person 5: [0 1 3 4 5]
person 6: [1 3 4 6]
person 7: [1 2 3 4 7]
